In [1]:
CONFIG = {

   
    "layer_points": {
        "technical_substance": 28,
        "jd_skill_alignment": 22,
        "production_engineering": 15,
        "behavioral_hireability": 12,
        "career_progression": 10,
        "company_context": 6,
        "education": 4,
    },
    "risk_penalty_max": 10,   
  
    "technical_substance_points": {
        "retrieval_search": 7,
        "embeddings": 5,
        "vector_databases": 4,
        "ranking_reranking": 4,
        "llm_finetuning": 3,
        "eval_frameworks": 3,      # NDCG, MAP, MRR
        "distributed_ml_inference": 2,
    },

    
    "jd_skill_points": {
        "python": 3,
        "retrieval": 3,
        "vector db": 3,
        "llm fine-tuning": 2,
        "milvus": 2, "pinecone": 2, "qdrant": 2, "faiss": 2,  
        "ranking": 2,
        "evaluation metrics": 2,
        "spark": 1,
        "airflow": 1,
        "docker": 1, "kubernetes": 1,   
        "github": 1, "open source": 1,  
        "sql": 1,
    },

    
    "production_engineering_points": {
        "built_production_system": 4,
        "owned_architecture": 3,
        "deployed_to_users": 3,
        "current_hands_on_engineer": 2,
        "scaling_optimization": 2,
        "monitoring_evaluation": 1,
    },

   
    "behavioral_points": {
        "open_to_work": 2,
        "recruiter_response_rate": 2,
        "interview_completion_rate": 2,
        "github_activity": 2,
        "recently_active": 1,
        "search_appearances": 1,
        "saved_by_recruiters": 1,
        "profile_completeness": 1,
    },


    "career_progression_points": {
        "years_5_to_9": 3,
        "promotions_growth": 2,
        "stable_tenure": 2,
        "recent_engineering_role": 2,
        "leadership_mentoring": 1,
    },

  
    "company_context_points": {
        "ai_product_company": 3,
        "product_engineering": 2,
        "large_scale_systems_exposure": 1,
    },

    
    "education_points": {
        "cs_ai_ml_degree": 2,
        "tier_1_2_institute": 1,
        "relevant_higher_studies": 1,
    },

    
    "risk_penalty_points": {
        "keyword_stuffing": -3,
        "contradictory_career_history": -3,
        "impossible_timeline_soft": -2,   
        "extremely_long_notice_period": -1,
        "no_recent_activity": -1,
        "self_rated_skill_exceeds_measured": -2,
    },

    "consistency_penalty": {
        "enabled": True,
        "gap_threshold": 20,
        "severe_gap_threshold": 35,
    },

    "honeypot_rules": {
        "max_plausible_age_for_experience": True,
        "min_working_age": 18,
        "max_working_age": 70,
        "check_overlapping_roles": True,
        "check_degree_year_consistency": True,
        "max_allowed_simultaneous_current_roles": 1,
    },


    "proficiency_scale": {
        "beginner": 25,
        "intermediate": 55,
        "advanced": 85,
    },


    "top_n": 100,
    "output_columns": ["candidate_id", "rank", "score", "reasoning"],
}


_layer_sum = sum(CONFIG["layer_points"].values())

print(f"7 positive layers sum to: {_layer_sum} (Doc 4 summary-row target was 100; ""we use the detailed penalty table instead -- see note above)")
assert _layer_sum == 97, f"Positive layers must sum to 97 given current layer_points, got {_layer_sum}"

_expected_header = ["candidate_id", "rank", "score", "reasoning"]
assert CONFIG["output_columns"] == _expected_header, (
    f"output_columns must be {_expected_header}, got {CONFIG['output_columns']}"
)

print(f"Risk penalty max: -{CONFIG['risk_penalty_max']} "
      f"(sum of risk_penalty_points: {sum(CONFIG['risk_penalty_points'].values())})")
print("\nLayer budgets:")
for k, v in CONFIG["layer_points"].items():
    print(f"  {k}: {v}")

7 positive layers sum to: 97 (Doc 4 summary-row target was 100; we use the detailed penalty table instead -- see note above)
Risk penalty max: -10 (sum of risk_penalty_points: -12)

Layer budgets:
  technical_substance: 28
  jd_skill_alignment: 22
  production_engineering: 15
  behavioral_hireability: 12
  career_progression: 10
  company_context: 6
  education: 4


In [2]:
import json
import gzip
import csv
import time
import io
import os
from datetime import datetime

CANDIDATES_PATH = "candidates.jsonl"     
JD_PATH = "job_description.txt"   


def _open_path(path):
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "rt", encoding="utf-8")


def _base_name(path):
    p = path[:-3] if path.endswith(".gz") else path
    return p.lower()


def wait_until_file_stable(path, checks=3, interval=4):
    """
        Returns once the file size stops changing across `checks` consecutive reads `interval` seconds apart -- i.e. the upload/write has finished. Prevents loading a partially-uploaded file.
    """

    if not os.path.exists(path):
        raise RuntimeError(f"{path} does not exist. Upload it first.")
    last = -1
    stable_count = 0
    while stable_count < checks:
        size = os.path.getsize(path)
        if size == last:
            stable_count += 1
        else:
            stable_count = 0
        last = size
        if stable_count < checks:
            time.sleep(interval)
    print(f"  File stable at {last/1e6:.1f} MB -- upload complete.")
    return last


def load_candidates_from_path(path):
    """
        Reads all candidates. For .jsonl, parses each line independently and skips (with a warning) any malformed line rather than crashing.
    """

    base = _base_name(path)
    t0 = time.time()
    records = []
    bad_lines = []

    if base.endswith(".jsonl"):
        with _open_path(path) as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as e:
                    bad_lines.append((line_num, str(e)))
    elif base.endswith(".json"):
        with _open_path(path) as f:
            parsed = json.load(f)
        records = parsed if isinstance(parsed, list) else [parsed]
    elif base.endswith(".csv"):
        with _open_path(path) as f:
            records = [dict(row) for row in csv.DictReader(f)]
    else:
        with _open_path(path) as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as e:
                    bad_lines.append((line_num, str(e)))

    print(f"Loaded {len(records):,} candidate records from {path} in {time.time()-t0:.2f}s")
    if bad_lines:
        print(f"  WARNING: skipped {len(bad_lines)} malformed line(s). First few:")
        for ln, err in bad_lines[:3]:
            print(f"    line {ln}: {err}")
    if not records:
        raise RuntimeError(f"{path} parsed to 0 records -> check the file.")
    CONFIG["candidates_file_name"] = path
    return records


def load_jd_from_path(path):
    base = _base_name(path)
    with _open_path(path) as f:
        text = f.read()
    if base.endswith(".json"):
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            text = parsed.get("description") or parsed.get("job_description") or text
    if not text or not text.strip():
        raise RuntimeError(f"{path} parsed to empty JD text -> check the file.")
    CONFIG["jd_file_name"] = path
    return text



print("Checking candidates file is fully uploaded...")
wait_until_file_stable(CANDIDATES_PATH)  

all_candidates = load_candidates_from_path(CANDIDATES_PATH)

if os.path.exists(JD_PATH):
    jd_text = load_jd_from_path(JD_PATH)
else:
    print(f"NOTE: {JD_PATH} not found -- using inline JD text. Point JD_PATH at your file to override.")
    jd_text = (
        "Senior AI Engineer - Founding Team, Redrob AI. "
        "Own AI/ML systems end-to-end in production. Experience shipping and owning "
        "real ML/AI products is critical. Skills in NLP, LLMs, and modern AI tooling "
        "are valued, but demonstrated ownership, production judgment, and career "
        "trajectory matter far more than a long list of keywords."
    )

_first_candidate = all_candidates[0]

print(f"\nJD loaded ({len(jd_text)} chars)")
print(f"First candidate: {_first_candidate.get('candidate_id', '<no id field>')}")
print(f"Total candidates in memory: {len(all_candidates):,}")


def _print_keys(label, d):
    if isinstance(d, dict):
        print(f"  {label}: {sorted(d.keys())}")
    else:
        print(f"  {label}: <not a dict, got {type(d).__name__}>")

print("\n--- SCHEMA CHECK (first candidate) ---")
_c = _first_candidate
print("Top-level keys:", sorted(_c.keys()))
_print_keys("profile", _c.get("profile", {}))
_print_keys("redrob_signals", _c.get("redrob_signals", {}))

_hist = _c.get("career_history", []) or []
if _hist:
    print(f"  career_history[0] keys: {sorted(_hist[0].keys())}")
    print(f"  career_history[0] sample values: "
          f"industry={_hist[0].get('industry')!r}, company={_hist[0].get('company')!r}")
else:
    print("  career_history: EMPTY for this candidate")

_edu = _c.get("education", []) or []
if _edu:
    print(f"  education[0] keys: {sorted(_edu[0].keys())}")
    print(f"  education[0] sample: {_edu[0]}")
else:
    print("  education: EMPTY for this candidate")

_skills = _c.get("skills", []) or []
if _skills:
    print(f"  skills[0] keys: {sorted(_skills[0].keys())}")

Checking candidates file is fully uploaded...
  File stable at 487.3 MB -- upload complete.
Loaded 100,000 candidate records from candidates.jsonl in 25.09s

JD loaded (9632 chars)
First candidate: CAND_0000001
Total candidates in memory: 100,000

--- SCHEMA CHECK (first candidate) ---
Top-level keys: ['candidate_id', 'career_history', 'certifications', 'education', 'languages', 'profile', 'redrob_signals', 'skills']
  profile: ['anonymized_name', 'country', 'current_company', 'current_company_size', 'current_industry', 'current_title', 'headline', 'location', 'summary', 'years_of_experience']
  redrob_signals: ['applications_submitted_30d', 'avg_response_time_hours', 'connection_count', 'endorsements_received', 'expected_salary_range_inr_lpa', 'github_activity_score', 'interview_completion_rate', 'last_active_date', 'linkedin_connected', 'notice_period_days', 'offer_acceptance_rate', 'open_to_work_flag', 'preferred_work_mode', 'profile_completeness_score', 'profile_views_received_30d'

In [3]:
from datetime import datetime

DATE_FMT = "%Y-%m-%d"

REFERENCE_DATE = datetime(2026, 6, 30)


def _parse_date(date_str):
    if not date_str:
        return None
    try:
        return datetime.strptime(date_str, DATE_FMT)
    except (ValueError, TypeError):
        return None


def check_experience_impossible(candidate, rules):
    """
        Flags experience that is physically IMPOSSIBLE, not merely larger than the listed jobs.
    """
    reasons = []
    profile = candidate.get("profile", {}) or {}
    years_exp = profile.get("years_of_experience")
    if years_exp is None:
        return reasons

    
    if years_exp > 60:
        reasons.append(f"years_of_experience ({years_exp}) exceeds a plausible human maximum")
        return reasons

    
    education = candidate.get("education", []) or []
    degree_end_years = [e.get("end_year") for e in education if e.get("end_year")]
    if degree_end_years:
        earliest_degree_end = min(degree_end_years)
        now_year = REFERENCE_DATE.year
        implied_work_start_year = now_year - years_exp
        
        if implied_work_start_year < earliest_degree_end - 6:
            reasons.append(
                f"years_of_experience ({years_exp}) implies work starting in "
                f"~{implied_work_start_year:.0f}, more than 6 years before degree "
                f"completion ({earliest_degree_end}) -- implausible"
            )

    
    history = candidate.get("career_history", []) or []
    starts = [_parse_date(h.get("start_date")) for h in history]
    starts = [d for d in starts if d]
    if starts:
        span_years = (REFERENCE_DATE - min(starts)).days / 365.25
        if years_exp > span_years + 8:
            reasons.append(
                f"claims {years_exp}y experience but the entire listed career "
                f"spans only {span_years:.1f}y -- honeypot pattern"
            )
    return reasons


def check_overlapping_roles(candidate, rules):
    """
        Flags two non-current roles with overlapping date ranges.
    """

    reasons = []
    history = candidate.get("career_history", []) or []
    intervals = []
    for h in history:
        if h.get("is_current"):
            continue
        start = _parse_date(h.get("start_date"))
        end = _parse_date(h.get("end_date"))
        if start and end:
            intervals.append((start, end, h.get("company", "unknown")))

    intervals.sort(key=lambda x: x[0])
    for i in range(len(intervals) - 1):
        cur_start, cur_end, cur_company = intervals[i]
        next_start, next_end, next_company = intervals[i + 1]
        overlap_days = (cur_end - next_start).days
        if overlap_days > 31:  
            reasons.append(
                f"overlapping roles: {cur_company} ends {cur_end.date()} but "
                f"{next_company} starts {next_start.date()} ({overlap_days} days overlap)"
            )
    return reasons


def check_multiple_current_roles(candidate, rules):
    """
        Flags more than the allowed number of simultaneous is_current roles.
    """

    reasons = []
    history = candidate.get("career_history", []) or []
    current_count = sum(1 for h in history if h.get("is_current"))
    max_allowed = rules.get("max_allowed_simultaneous_current_roles", 1)
    if current_count > max_allowed:
        reasons.append(f"{current_count} simultaneous current roles (max allowed: {max_allowed})")
    return reasons


def check_degree_after_career(candidate, rules):
    """
        Flags the impossible case where the latest job ends before the degree even started. Does NOT flag normal internships-during-study.
    """

    reasons = []
    education = candidate.get("education", []) or []
    history = candidate.get("career_history", []) or []
    if not education or not history:
        return reasons

    degree_start_years = [e.get("start_year") for e in education if e.get("start_year")]
    if not degree_start_years:
        return reasons
    earliest_degree_start = min(degree_start_years)

    end_dates = []
    for h in history:
        if h.get("is_current"):
            end_dates.append(REFERENCE_DATE)
        else:
            d = _parse_date(h.get("end_date"))
            if d:
                end_dates.append(d)
    if not end_dates:
        return reasons
    latest_job_end = max(end_dates)

    if latest_job_end.year < earliest_degree_start - 1:
        reasons.append(
            f"latest job ends {latest_job_end.year} but degree only starts "
            f"{earliest_degree_start} -- impossible timeline"
        )
    return reasons


def check_expert_skills_never_used(candidate, rules):
    """
        Spec-named honeypot: 'expert' proficiency in skills with 0 years used. Flags only an EXPLICIT duration_months == 0 (missing field is not 0) on 3+ expert skills. Only 21 of 100,000 profiles match -- safe hard gate.
    """
    
    zero = sum(
        1 for s in (candidate.get("skills", []) or [])
        if s.get("proficiency") == "expert" and s.get("duration_months") == 0
    )
    if zero >= 3:
        return [f"{zero} skills self-rated 'expert' with an explicit 0 months of use"]
    return []


def run_honeypot_gate(candidate, config):
    """
        Returns (is_valid, all_reasons). is_valid=False means DROP before scoring.
    """
    rules = config["honeypot_rules"]
    all_reasons = []

    all_reasons += check_experience_impossible(candidate, rules)
    if rules.get("check_overlapping_roles"):
        all_reasons += check_overlapping_roles(candidate, rules)
    all_reasons += check_multiple_current_roles(candidate, rules)
    all_reasons += check_degree_after_career(candidate, rules)
    all_reasons += check_expert_skills_never_used(candidate, rules)

    is_valid = len(all_reasons) == 0
    return is_valid, all_reasons



_realistic = {
    "profile": {"years_of_experience": 10.0},
    "career_history": [
        {"start_date": "2022-01-01", "end_date": None, "is_current": True, "duration_months": 42},
        {"start_date": "2019-06-01", "end_date": "2021-12-01", "is_current": False, "duration_months": 30},
    ],
    "education": [{"end_year": 2016, "start_year": 2012}],
}
_impossible = {
    "profile": {"years_of_experience": 15.0},
    "career_history": [{"start_date": "2023-01-01", "end_date": None, "is_current": True, "duration_months": 12}],
    "education": [{"end_year": 2023, "start_year": 2019}],
}
print("Realistic (10y exp, listed 7y):", run_honeypot_gate(_realistic, CONFIG))
print("Impossible (15y exp, degree 2023):", run_honeypot_gate(_impossible, CONFIG))

Realistic (10y exp, listed 7y): (True, [])
Impossible (15y exp, degree 2023): (False, ['years_of_experience (15.0) implies work starting in ~2011, more than 6 years before degree completion (2023) -- implausible', 'claims 15.0y experience but the entire listed career spans only 3.5y -- honeypot pattern'])


In [4]:
try:
    from rank_bm25 import BM25Okapi
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rank-bm25"], check=True)
    from rank_bm25 import BM25Okapi

import re
import numpy as np

PRODUCTION_REFERENCE_TEXT = (
    "Owned and shipped production systems end to end. Deployed and scaled "
    "services serving real traffic. On-call ownership of reliability and "
    "incidents. Took systems from design to production."
)

_STOP = {"the","and","for","are","but","with","you","your","our","who","that","this",
         "have","has","not","will","can","far","more","than","long","list","about",
         "into","real","want","looking","team","a","an","of","to","in","on","is","as"}

def _tokenize(text):
    toks = re.findall(r"[a-zA-Z][a-zA-Z+#.]{1,}", (text or "").lower())
    return [t for t in toks if t not in _STOP]


BM25_INDEX = None
JD_TOKENS = None
PROD_REF_TOKENS = None


def fit_bm25(jd_text, all_texts):
    """
        Build a BM25 index over all candidate docs; populate globals.
    """
    global BM25_INDEX, JD_TOKENS, PROD_REF_TOKENS
    tokenized_docs = [_tokenize(t) for t in all_texts]
    BM25_INDEX = BM25Okapi(tokenized_docs)
    JD_TOKENS = _tokenize(jd_text)
    PROD_REF_TOKENS = _tokenize(PRODUCTION_REFERENCE_TEXT)


def bm25_scores_vs_jd():
    return np.asarray(BM25_INDEX.get_scores(JD_TOKENS), dtype=float)


def bm25_scores_vs_prodref():
    return np.asarray(BM25_INDEX.get_scores(PROD_REF_TOKENS), dtype=float)


print("BM25 setup loaded. No model download / no GPU / no internet -- pure CPU.")

BM25 setup loaded. No model download / no GPU / no internet -- pure CPU.


In [5]:
TECHNICAL_SUBSTANCE_PHRASES = {
    "retrieval_search": [
        "retrieval", "search system", "search engine", "information retrieval",
        "semantic search", "query understanding", "recommendation system",
        "candidate retrieval", "document retrieval", "search infrastructure",
        "search relevance", "query", "indexing", "recommender",
    ],
    "embeddings": [
        "embedding", "embeddings", "vector representation", "sentence embedding",
        "dense retrieval", "text embedding", "embedding model", "encoder model",
        "semantic representation",
    ],
    "vector_databases": [
        "vector database", "milvus", "faiss", "pinecone", "qdrant", "weaviate",
        "vector store", "vector index", "ann search", "approximate nearest neighbor",
        "pgvector", "nearest neighbor search",
    ],
    "ranking_reranking": [
        "ranking", "re-ranking", "reranking", "learning to rank", "relevance ranking",
        "rank model", "scoring model", "candidate ranking", "ranked results",
        "ranking pipeline", "ranking algorithm",
    ],
    "llm_finetuning": [
        "fine-tuning", "fine tuned", "finetune", "lora", "peft", "instruction tuning",
        "rlhf", "llm", "large language model", "prompt engineering", "model tuning",
        "domain adaptation",
    ],
    "eval_frameworks": [
        "ndcg", "map@", "mrr", "precision@", "recall@", "evaluation framework",
        "offline evaluation", "a/b test", "ab test", "eval pipeline", "evaluation metrics",
        "benchmarking", "model evaluation",
    ],
    "distributed_ml_inference": [
        "distributed training", "model inference", "inference pipeline",
        "distributed systems", "gpu cluster", "model serving", "low latency",
        "high throughput", "distributed computing", "parallel processing",
        "scalable inference",
    ],
}


_SATURATION_HITS = 2


def _technical_text(candidate):
    """
        Career descriptions + summary ONLY -- deliberately excludes the skills list, which is scored separately in Layer 3.
    """
    profile = candidate.get("profile", {}) or {}
    summary = profile.get("summary", "") or ""
    history = candidate.get("career_history", []) or []
    role_bits = [(r.get("title", "") or "") + ". " + (r.get("description", "") or "") for r in history]
    return (summary + " " + " ".join(role_bits)).lower()


def score_technical_substance(candidate, config):
    """
        Returns (score_0_to_28, evidence_dict).
    """
    text = _technical_text(candidate)
    points_table = config["technical_substance_points"]

    breakdown = {}
    total = 0.0
    for category, max_points in points_table.items():
        phrases = TECHNICAL_SUBSTANCE_PHRASES.get(category, [])
        hits = [p for p in phrases if p in text]
        n_hits = len(hits)
        fraction = min(1.0, n_hits / _SATURATION_HITS) if phrases else 0.0
        earned = round(max_points * fraction, 2)
        total += earned
        breakdown[category] = {"earned": earned, "max": max_points, "hits": hits}

    total = min(total, config["layer_points"]["technical_substance"])
    return total, {"breakdown": breakdown}


print("Layer 2 (Technical Substance, 28 pts) loaded.")

Layer 2 (Technical Substance, 28 pts) loaded.


In [6]:
OWNERSHIP_VERBS = [
    "owned", "own ", "led", "leading", "built and shipped", "shipped",
    "deployed", "architected", "designed and built", "drove", "spearheaded",
    "on-call", "on call", "end to end", "end-to-end", "from scratch",
    "took ownership", "responsible for", "maintained", "scaled",
]

HEDGE_WORDS = [
    "exposure", "exposed to", "assisted", "supported the", "helped the",
    "familiar with", "worked closely with", "adjacent", "some exposure",
    "involved in", "contributed to", "learning", "building competence",
    "interested in", "transitioning",
]


_JD_SKILL_STANDALONE = {
    "python":             (["python"], 3),
    "retrieval":          (["retrieval", "information retrieval", "semantic search", "rag", "dense retrieval"], 3),
    "vector_db":          (["vector search", "pgvector", "pinecone", "qdrant", "faiss", "milvus", "weaviate", "opensearch", "vector database"], 3),
    "llm_finetuning":     (["fine-tuning llms", "lora", "qlora", "peft", "instruction tuning", "rlhf", "fine-tuning", "finetuning"], 2),
    "ranking":            (["ranking", "learning to rank", "bm25", "recommendation systems", "reranking", "re-ranking"], 2),
    "evaluation_metrics": (["ndcg", "mrr", "map@", "precision@", "recall@", "evaluation metrics", "offline evaluation"], 2),
    "spark":              (["spark", "pyspark"], 1),
    "airflow":            (["airflow"], 1),
    "sql":                (["sql", "postgresql", "snowflake", "dbt"], 1),
}

_JD_SKILL_GROUPS = {
    "vector_db_tools": (["milvus", "pinecone", "qdrant", "faiss", "weaviate", "pgvector", "opensearch"], 2),
    "container_tools": (["docker", "kubernetes", "k8s"], 1),
    "github_oss":      (["github", "open source", "open-source"], 1),
}



def _skills_and_summary_text(candidate):
    skills = candidate.get("skills", []) or []
    skill_names = [s.get("name", "") or "" for s in skills]
    summary = (candidate.get("profile", {}) or {}).get("summary", "") or ""
   
    return (" | ".join(skill_names) + " | " + summary).lower()


def score_jd_skill_alignment(candidate, config):
    """
        Returns (score_0_to_22, evidence_dict). Alias-aware: each JD skill matches against the real dataset vocabulary. Grouping logic isn't a flat dict, but the total budget matches config['jd_skill_points'].
    """
    text = _skills_and_summary_text(candidate)

    matched = {}
    missing_key = []   
    total = 0.0
    _KEY_SKILLS = {"python", "retrieval", "vector_db", "llm_finetuning", "ranking", "evaluation_metrics"}
    
    for key, (aliases, pts) in _JD_SKILL_STANDALONE.items():
        hit = next((a for a in aliases if a in text), None)
        if hit:
            matched[hit] = pts
            total += pts
        elif key in _KEY_SKILLS:
            missing_key.append(key)

    
    for group_name, (terms, pts) in _JD_SKILL_GROUPS.items():
        hit = next((t for t in terms if t in text), None)
        if hit:
            if hit not in matched:
                matched[hit] = pts
            total += pts

    max_points = config["layer_points"]["jd_skill_alignment"]
    total = min(total, max_points)
    return total, {"matched": matched, "missing_key": missing_key}

print("Layer 3 (JD Skill Alignment, 22 pts) loaded. OWNERSHIP_VERBS/HEDGE_WORDS ready for Layer 4.")

Layer 3 (JD Skill Alignment, 22 pts) loaded. OWNERSHIP_VERBS/HEDGE_WORDS ready for Layer 4.


In [7]:
PRODUCTION_PHRASES = {
    "built_production_system": ["built", "shipped", "built and shipped", "developed and launched"],
    "owned_architecture": ["owned", "architected", "designed and built", "took ownership", "system design"],
    "deployed_to_users": ["deployed", "launched", "released to production", "serving real traffic",
                           "in production", "live in production"],
    "scaling_optimization": ["scaled", "scaling", "optimized", "performance improvement",
                              "latency reduction", "throughput", "cost reduction"],
    "monitoring_evaluation": ["on-call", "on call", "monitoring", "observability",
                               "incident response", "reliability", "alerting"],
}


def _role_texts(candidate):
    """
        Returns list of (title, description_lowercased) per role, plus summary.
    """
    history = candidate.get("career_history", []) or []
    return [((r.get("title", "") or ""), (r.get("description", "") or "").lower()) for r in history]


def score_production_engineering(candidate, config):
    """
        Returns (score_0_to_15, evidence_dict).
    """
    points_table = config["production_engineering_points"]
    roles = _role_texts(candidate)
    full_text = " ".join(desc for _, desc in roles)

    breakdown = {}
    total = 0.0

    
    for category, max_points in points_table.items():
        if category == "current_hands_on_engineer":
            continue  
        phrases = PRODUCTION_PHRASES.get(category, [])
        hit = any(p in full_text for p in phrases)
        if not hit:
            breakdown[category] = {"earned": 0.0, "max": max_points}
            continue
        
        hedged = any(h in full_text for h in HEDGE_WORDS)
        earned = max_points * (0.5 if hedged else 1.0)
        total += earned
        breakdown[category] = {"earned": round(earned, 2), "max": max_points, "hedged": hedged}

    
    max_hands_on = points_table.get("current_hands_on_engineer", 2)
    history = candidate.get("career_history", []) or []
    current_roles = [r for r in history if r.get("is_current")]
    hands_on_earned = 0.0
    if current_roles:
        cur = current_roles[0]
        title = (cur.get("title", "") or "").lower()
        desc = (cur.get("description", "") or "").lower()
        is_pure_mgmt = any(k in title for k in ["director", "vp ", "vice president", "head of"])
        has_ownership = any(v in desc for v in OWNERSHIP_VERBS)
        if not is_pure_mgmt and has_ownership:
            hands_on_earned = max_hands_on
        elif not is_pure_mgmt:
            hands_on_earned = max_hands_on * 0.5
    total += hands_on_earned
    breakdown["current_hands_on_engineer"] = {"earned": round(hands_on_earned, 2), "max": max_hands_on}

    max_points_total = config["layer_points"]["production_engineering"]
    total = min(total, max_points_total)
    return total, {"breakdown": breakdown}


print("Layer 4 (Production Engineering, 15 pts) loaded.")

Layer 4 (Production Engineering, 15 pts) loaded.


In [8]:
from datetime import datetime as _dt_bh

_MISSING_BEHAVIORAL_FIELDS = set()  


def _safe_rate(val):
    """
        Coerces a 0..1 rate-like value; returns None if not usable.
    """
    if isinstance(val, (int, float)) and 0.0 <= val <= 1.0:
        return float(val)
    if isinstance(val, (int, float)) and 1.0 < val <= 100.0:
        return float(val) / 100.0
    return None


def score_behavioral_hireability(candidate, config, now_str="2026-06-30"):
    """
        Returns (score_0_to_12, evidence_dict).
    """
    points_table = config["behavioral_points"]
    sig = candidate.get("redrob_signals", {}) or {}
    breakdown = {}
    total = 0.0

    
    pts = points_table["open_to_work"]
    val = sig.get("open_to_work_flag")
    if val is None:
        _MISSING_BEHAVIORAL_FIELDS.add("open_to_work_flag")
        earned = pts * 0.5
    else:
        earned = pts if val else 0.0
    total += earned
    breakdown["open_to_work"] = {"earned": earned, "max": pts, "raw": val}

    
    pts = points_table["recruiter_response_rate"]
    rate = _safe_rate(sig.get("recruiter_response_rate"))
    earned = pts * rate if rate is not None else 0.0
    if rate is None: _MISSING_BEHAVIORAL_FIELDS.add("recruiter_response_rate")
    total += earned
    breakdown["recruiter_response_rate"] = {"earned": round(earned, 2), "max": pts, "raw": sig.get("recruiter_response_rate")}

   
    pts = points_table["interview_completion_rate"]
    rate = _safe_rate(sig.get("interview_completion_rate"))
    earned = pts * rate if rate is not None else 0.0
    if rate is None: _MISSING_BEHAVIORAL_FIELDS.add("interview_completion_rate")
    total += earned
    breakdown["interview_completion_rate"] = {"earned": round(earned, 2), "max": pts, "raw": sig.get("interview_completion_rate")}

    
    pts = points_table["github_activity"]
    gh = sig.get("github_activity_score")
    if gh is None:
        _MISSING_BEHAVIORAL_FIELDS.add("github_activity_score")
        earned = pts * 0.5
    elif isinstance(gh, (int, float)) and gh <= 1.0:
        earned = pts * gh
    elif isinstance(gh, (int, float)):
        
        earned = pts * min(1.0, gh / 100.0)
    else:
        earned = pts * 0.5
    total += earned
    breakdown["github_activity"] = {"earned": round(earned, 2), "max": pts, "raw": gh}

    
    pts = points_table["recently_active"]
    la = sig.get("last_active_date")
    earned = 0.0
    try:
        days = (_dt_bh.strptime(now_str, "%Y-%m-%d") - _dt_bh.strptime(la, "%Y-%m-%d")).days
        if days <= 30:
            earned = pts
        elif days <= 90:
            earned = pts * 0.5
    except (TypeError, ValueError):
        _MISSING_BEHAVIORAL_FIELDS.add("last_active_date")
    total += earned
    breakdown["recently_active"] = {"earned": earned, "max": pts, "raw": la}

    
    pts = points_table["search_appearances"]
    sa = sig.get("search_appearance_30d")
    if sa is None:
        _MISSING_BEHAVIORAL_FIELDS.add("search_appearance_30d")
        earned = pts * 0.5
    else:
        
        earned = pts * min(1.0, float(sa) / 105.0) if isinstance(sa, (int, float)) else pts * 0.5
    total += earned
    breakdown["search_appearances"] = {"earned": round(earned, 2), "max": pts, "raw": sa}

    
    pts = points_table["saved_by_recruiters"]
    sv = sig.get("saved_by_recruiters_30d")
    if sv is None:
        _MISSING_BEHAVIORAL_FIELDS.add("saved_by_recruiters_30d")
        earned = pts * 0.5
    else:
        
        earned = pts * min(1.0, float(sv) / 5.0) if isinstance(sv, (int, float)) else (pts if sv else 0.0)
    total += earned
    breakdown["saved_by_recruiters"] = {"earned": round(earned, 2), "max": pts, "raw": sv}

    
    pts = points_table["profile_completeness"]
    pc_raw = sig.get("profile_completeness_score")
    if pc_raw is None:
        _MISSING_BEHAVIORAL_FIELDS.add("profile_completeness_score")
        earned = pts * 0.5
    elif isinstance(pc_raw, (int, float)):
        
        earned = pts * min(1.0, pc_raw / 100.0)
    else:
        earned = pts * 0.5
    total += earned
    breakdown["profile_completeness"] = {"earned": round(earned, 2), "max": pts, "raw": pc_raw}

    max_points = config["layer_points"]["behavioral_hireability"]
    total = min(total, max_points)
    return total, {"breakdown": breakdown}


print("Layer 5 (Behavioral & Hireability, 12 pts) loaded.")
print("NOTE: run this after Cell 2's schema check -- guessed fields not found will")
print("print a warning once you run score_behavioral_hireability on real data")
print("(check _MISSING_BEHAVIORAL_FIELDS after a test run).")

Layer 5 (Behavioral & Hireability, 12 pts) loaded.
NOTE: run this after Cell 2's schema check -- guessed fields not found will
print a warning once you run score_behavioral_hireability on real data
(check _MISSING_BEHAVIORAL_FIELDS after a test run).


In [9]:
SENIORITY_LEVELS = [
    (6, ["chief", "cto", "vp ", "vice president", "head of", "director"]),
    (5, ["principal", "staff", "distinguished"]),
    (4, ["lead", "manager", "founding", "founder"]),
    (3, ["senior", "sr.", "sr "]),
    (2, ["engineer", "developer", "scientist", "analyst", "specialist"]),
    (1, ["junior", "jr.", "jr ", "intern", "trainee", "associate", "graduate"]),
]

_TECH_TITLE_TERMS = [
    "engineer", "developer", "scientist", "researcher", "architect",
    "ml", "ai", "data", "software", "backend", "infrastructure",
]

_LEADERSHIP_TERMS = [
    "mentored", "mentor", "led a team", "managed engineers", "team lead",
    "manager", "led the team", "coached", "supervised",
]


def _title_seniority(title):
    if not title:
        return 2
    tl = title.lower()
    for level, keywords in SENIORITY_LEVELS:
        if any(k in tl for k in keywords):
            return level
    return 2


def score_career_progression(candidate, config):
    """
        Returns (score_0_to_10, evidence_dict).
    """

    points_table = config["career_progression_points"]
    profile = candidate.get("profile", {}) or {}
    history = candidate.get("career_history", []) or []
    breakdown = {}
    total = 0.0

    # --- years_5_to_9: triangular credit peaked in [5,9] ---
    pts = points_table["years_5_to_9"]
    years = profile.get("years_of_experience")
    earned = 0.0
    if isinstance(years, (int, float)):
        if 5 <= years <= 9:
            earned = pts
        elif 3 <= years < 5:
            earned = pts * (years - 3) / 2.0
        elif 9 < years <= 12:
            earned = pts * (12 - years) / 3.0
    total += earned
    breakdown["years_5_to_9"] = {"earned": round(earned, 2), "max": pts, "years": years}

    if history:
        def _start_key(r):
            d = _parse_date(r.get("start_date"))
            return d if d is not None else datetime.min
        ordered = sorted(history, key=_start_key)
        levels = [_title_seniority(r.get("title")) for r in ordered]
        net_growth = levels[-1] - levels[0]
    else:
        levels, net_growth = [], 0

    
    pts = points_table["promotions_growth"]
    if net_growth >= 2: earned = pts
    elif net_growth == 1: earned = pts * 0.75
    elif net_growth == 0: earned = pts * 0.25
    else: earned = 0.0
    total += earned
    breakdown["promotions_growth"] = {"earned": round(earned, 2), "max": pts, "net_growth": net_growth}

    
    pts = points_table["stable_tenure"]
    past = [r for r in history if not r.get("is_current")]
    if past:
        avg_months = sum(r.get("duration_months", 0) or 0 for r in past) / len(past)
    else:
        avg_months = 0
    if avg_months >= 18: earned = pts
    elif avg_months >= 12: earned = pts * 0.5
    else: earned = 0.0
    total += earned
    breakdown["stable_tenure"] = {"earned": round(earned, 2), "max": pts, "avg_months": round(avg_months, 1)}

    
    pts = points_table["recent_engineering_role"]
    current = [r for r in history if r.get("is_current")]
    ref_role = current[0] if current else (sorted(history, key=_start_key, reverse=True)[0] if history else None)
    earned = 0.0
    if ref_role:
        title = (ref_role.get("title", "") or "").lower()
        if any(t in title for t in _TECH_TITLE_TERMS):
            earned = pts
    total += earned
    breakdown["recent_engineering_role"] = {"earned": earned, "max": pts}

    
    pts = points_table["leadership_mentoring"]
    full_text = " ".join(((r.get("title", "") or "") + " " + (r.get("description", "") or "")) for r in history).lower()
    earned = pts if any(t in full_text for t in _LEADERSHIP_TERMS) else 0.0
    total += earned
    breakdown["leadership_mentoring"] = {"earned": earned, "max": pts}

    max_points = config["layer_points"]["career_progression"]
    total = min(total, max_points)
    return total, {"breakdown": breakdown}


print("Layer 6 (Career Progression, 10 pts) loaded.")

Layer 6 (Career Progression, 10 pts) loaded.


In [10]:
_SERVICES_IND = {"IT Services", "Consulting"}
_SERVICES_CO = {"tcs", "infosys", "wipro", "accenture", "cognizant",
                 "capgemini", "tech mahindra", "hcl", "ltimindtree"}

_AI_PRODUCT_INDUSTRIES = {"Artificial Intelligence", "AI", "Machine Learning",
                           "Technology", "Software", "Internet", "SaaS", "AI/ML"}

_KNOWN_TIER_INSTITUTES = [
    "iit", "nit", "iiit", "bits pilani", "bits", "iisc", "vjti", "iim",
]

_CS_AI_ML_TERMS = ["computer science", "artificial intelligence", "machine learning",
                    "data science", "computer engineering", " cs ", " ai ", " ml "]
_ENGINEERING_ADJACENT_TERMS = ["electronics", "information technology", "electrical", "mathematics"]


def _current_or_latest_role(candidate):
    history = candidate.get("career_history", []) or []
    current = [r for r in history if r.get("is_current")]
    if current:
        return current[0]
    if history:
        def _start_key(r):
            d = _parse_date(r.get("start_date"))
            return d if d is not None else datetime.min
        return sorted(history, key=_start_key, reverse=True)[0]
    return None


def score_company_context(candidate, config):
    """
        Returns (score_0_to_6, evidence_dict).
    """

    points_table = config["company_context_points"]
    role = _current_or_latest_role(candidate)
    breakdown = {}
    total = 0.0

    if not role:
        return 0.0, {"reason": "no career history"}

    industry = role.get("industry", "") or ""
    company = (role.get("company", "") or "").lower()
    desc = (role.get("description", "") or "").lower()

    is_services = (industry in _SERVICES_IND) or any(s in company for s in _SERVICES_CO)

    
    pts = points_table["ai_product_company"]
    if is_services:
        earned = 0.0
    elif industry in _AI_PRODUCT_INDUSTRIES:
        earned = pts
    else:
        earned = pts * 0.5  
    total += earned
    breakdown["ai_product_company"] = {"earned": round(earned, 2), "max": pts, "industry": industry}

   
    pts = points_table["product_engineering"]
    company_size = (candidate.get("profile", {}) or {}).get("current_company_size") or role.get("company_size") or ""
    _MID_LARGE_SIZES = {"201-500", "501-1000", "1001-5000", "5001-10000", "10001+"}
    _SMALL_STARTUP_SIZES = {"11-50", "51-200"}
    if is_services:
        earned = 0.0
    elif company_size in _MID_LARGE_SIZES:
        earned = pts
    elif company_size in _SMALL_STARTUP_SIZES:
        earned = pts * 0.75  
    elif "product" in desc:
        earned = pts * 0.5
    else:
        earned = pts * 0.25
    total += earned
    breakdown["product_engineering"] = {"earned": round(earned, 2), "max": pts, "company_size": company_size}

   
    pts = points_table["large_scale_systems_exposure"]
    _LARGE_SIZES = {"1001-5000", "5001-10000", "10001+"}
    if company_size in _LARGE_SIZES:
        earned = pts
    else:
        scale_terms = ["scale", "millions of users", "high traffic", "large-scale", "large scale"]
        earned = pts if any(t in desc for t in scale_terms) else 0.0
    total += earned
    breakdown["large_scale_systems_exposure"] = {"earned": earned, "max": pts}

    max_points = config["layer_points"]["company_context"]
    total = min(total, max_points)
    return total, {"breakdown": breakdown, "is_services_company": is_services}


def _education_text(edu_entry):
    """
        Concatenates all string values in an education dict -- defensive against not knowing the exact field name (degree/field/major/course).
    """

    return " ".join(str(v) for v in edu_entry.values() if isinstance(v, str)).lower()


def score_education(candidate, config):
    """
        Returns (score_0_to_4, evidence_dict). Uses the explicit `tier` and `field_of_study` fields from each education entry (both confirmed in schema) instead of string-matching against institution names.
    """

    points_table = config["education_points"]
    education = candidate.get("education", []) or []
    breakdown = {}
    total = 0.0

    if not education:
        return 0.0, {"reason": "no education listed"}

    
    _CS_AI_ML_FIELDS = {"Computer Science", "Computer Engineering", "Artificial Intelligence",
                          "Machine Learning", "Data Science", "Information Technology"}
    _ENG_ADJACENT_FIELDS = {"Electronics", "Electrical Engineering", "Mathematics",
                              "Statistics", "Physics"}
    pts = points_table["cs_ai_ml_degree"]
    fields = {e.get("field_of_study", "") for e in education}
    if fields & _CS_AI_ML_FIELDS:
        earned = pts
    elif fields & _ENG_ADJACENT_FIELDS:
        earned = pts * 0.5
    else:
        earned = 0.0
    total += earned
    breakdown["cs_ai_ml_degree"] = {"earned": round(earned, 2), "max": pts,
                                      "fields": sorted(f for f in fields if f)}

    
    pts = points_table["tier_1_2_institute"]
    tiers = {e.get("tier", "") for e in education}
    if "tier_1" in tiers:
        earned = pts
    elif "tier_2" in tiers:
        earned = pts * 0.75
    else:
        earned = 0.0
    total += earned
    breakdown["tier_1_2_institute"] = {"earned": round(earned, 2), "max": pts,
                                         "tiers": sorted(t for t in tiers if t)}

    
    pts = points_table["relevant_higher_studies"]
    degrees_text = " ".join(str(e.get("degree", "")).lower() for e in education)
    has_masters = any(t in degrees_text for t in ["master", "m.tech", "m.s.", "m.sc",
                                                     "phd", "ph.d", "mba"])
    earned = pts if (len(education) >= 2 or has_masters) else 0.0
    total += earned
    breakdown["relevant_higher_studies"] = {"earned": earned, "max": pts}

    max_points = config["layer_points"]["education"]
    total = min(total, max_points)
    return total, {"breakdown": breakdown}


print("Layer 7 (Company Context, 6 pts) + Layer 8 (Education, 4 pts) loaded.")


Layer 7 (Company Context, 6 pts) + Layer 8 (Education, 4 pts) loaded.


In [11]:
from datetime import datetime as _dt_risk


def _keyword_stuffing_penalty(tech_score, skill_score, config):
    """
        If skills claim a lot (Layer 3 high) but the career-history evidence for it is thin (Layer 2 low), that's the classic stuffing pattern the challenge explicitly warns about.
    """
    
    max_skill = config["layer_points"]["jd_skill_alignment"]
    max_tech = config["layer_points"]["technical_substance"]
    if max_skill == 0:
        return 0.0, False
    skill_frac = skill_score / max_skill
    tech_frac = tech_score / max_tech if max_tech else 0.0
    flagged = skill_frac >= 0.5 and tech_frac <= 0.2
    penalty = config["risk_penalty_points"]["keyword_stuffing"] if flagged else 0.0
    return penalty, flagged


def _soft_overlap_penalty(candidate, config):
    """
        Small overlaps (<=31 days) that the hard honeypot gate deliberately ignores, but are still worth a soft point deduction.
    """

    history = candidate.get("career_history", []) or []
    intervals = []
    for h in history:
        if h.get("is_current"):
            continue
        start = _parse_date(h.get("start_date"))
        end = _parse_date(h.get("end_date"))
        if start and end:
            intervals.append((start, end))
    intervals.sort(key=lambda x: x[0])
    for i in range(len(intervals) - 1):
        _, cur_end = intervals[i]
        next_start, _ = intervals[i + 1]
        overlap_days = (cur_end - next_start).days
        if 0 < overlap_days <= 31:
            return config["risk_penalty_points"]["contradictory_career_history"], True
    return 0.0, False


def _soft_timeline_penalty(candidate, config):
    """
        Experience implying work started 3-6 years before degree completion -- plausible-but-tight, softer than the honeypot's >6-year hard cutoff.
    """

    profile = candidate.get("profile", {}) or {}
    years_exp = profile.get("years_of_experience")
    education = candidate.get("education", []) or []
    degree_end_years = [e.get("end_year") for e in education if e.get("end_year")]
    if years_exp is None or not degree_end_years:
        return 0.0, False
    earliest_degree_end = min(degree_end_years)
    now_year = REFERENCE_DATE.year
    implied_start = now_year - years_exp
    if earliest_degree_end - 6 <= implied_start < earliest_degree_end - 3:
        return config["risk_penalty_points"]["impossible_timeline_soft"], True
    return 0.0, False


def _notice_period_penalty(candidate, config):
    sig = candidate.get("redrob_signals", {}) or {}
    notice = sig.get("notice_period_days")
    if isinstance(notice, (int, float)) and notice > 90:
        return config["risk_penalty_points"]["extremely_long_notice_period"], True
    return 0.0, False


def _inactivity_penalty(candidate, config, now_str="2026-06-30"):
    sig = candidate.get("redrob_signals", {}) or {}
    la = sig.get("last_active_date")
    try:
        days = (_dt_risk.strptime(now_str, "%Y-%m-%d") - _dt_risk.strptime(la, "%Y-%m-%d")).days
    except (TypeError, ValueError):
        return 0.0, False
    if days > 150:
        return config["risk_penalty_points"]["no_recent_activity"], True
    return 0.0, False


def _consistency_penalty(candidate, config):
    """
        Self-rated proficiency vs measured skill_assessment_scores --kept from the original notebook as a named risk item.
    """

    cfg = config["consistency_penalty"]
    if not cfg.get("enabled"):
        return 0.0, False
    scale = config["proficiency_scale"]
    signals = candidate.get("redrob_signals", {}) or {}
    measured = signals.get("skill_assessment_scores", {}) or {}
    skills = candidate.get("skills", []) or []

    worst_gap = 0
    for s in skills:
        name = s.get("name")
        prof = (s.get("proficiency") or "").lower()
        if name in measured and prof in scale:
            gap = scale[prof] - measured[name]
            worst_gap = max(worst_gap, gap)

    if worst_gap >= cfg["severe_gap_threshold"]:
        return config["risk_penalty_points"]["self_rated_skill_exceeds_measured"], True
    elif worst_gap >= cfg["gap_threshold"]:
        return config["risk_penalty_points"]["self_rated_skill_exceeds_measured"] * 0.5, True
    return 0.0, False


def score_risk_penalty(candidate, config, tech_score, skill_score):
    """
        Returns (penalty_as_negative_number, evidence_dict). Penalty is clamped to [-risk_penalty_max, 0].
    """

    items = {}
    total = 0.0

    p, flagged = _keyword_stuffing_penalty(tech_score, skill_score, config)
    if flagged: items["keyword_stuffing"] = p; total += p

    p, flagged = _soft_overlap_penalty(candidate, config)
    if flagged: items["contradictory_career_history"] = p; total += p

    p, flagged = _soft_timeline_penalty(candidate, config)
    if flagged: items["impossible_timeline_soft"] = p; total += p

    p, flagged = _notice_period_penalty(candidate, config)
    if flagged: items["extremely_long_notice_period"] = p; total += p

    p, flagged = _inactivity_penalty(candidate, config)
    if flagged: items["no_recent_activity"] = p; total += p

    p, flagged = _consistency_penalty(candidate, config)
    if flagged: items["self_rated_skill_exceeds_measured"] = p; total += p

    max_penalty = -config["risk_penalty_max"]
    total = max(total, max_penalty)  # can't exceed -10
    return total, {"items": items}


def _build_reasoning_layered(candidate, subs, penalty_evidence):
    """
        Fact-grounded reasoning built from each candidate's ACTUAL matched evidence (real phrases, real skill names, real years/title) -- not a templated "strong X and strong Y" sentence. Judges sample reasoning and penalize generic/templated text (Stage 4 brief), so every clause here is sourced from a specific field on this specific candidate.
    """

    profile = candidate.get("profile", {}) or {}
    years = profile.get("years_of_experience")
    title = profile.get("current_title", "") or ""

    parts = []

    if isinstance(years, (int, float)) and title:
        parts.append(f"{years:.1f}y experience, currently {title}")
    elif title:
        parts.append(f"Currently {title}")
    elif isinstance(years, (int, float)):
        parts.append(f"{years:.1f}y experience")

    _TECH_LABEL = {
        "retrieval_search": "retrieval/search", "embeddings": "embeddings",
        "vector_databases": "vector databases", "ranking_reranking": "ranking",
        "llm_finetuning": "LLM fine-tuning", "eval_frameworks": "evaluation metrics",
        "distributed_ml_inference": "distributed ML/inference",
    }
    tech_breakdown = subs["technical_substance"]["evidence"].get("breakdown", {})
    tech_hits = [(v["earned"], cat, v["hits"]) for cat, v in tech_breakdown.items() if v.get("hits")]
    tech_hits.sort(key=lambda x: -x[0])
    if tech_hits:
        top = tech_hits[:2]
        labels = [_TECH_LABEL.get(c, c) for _, c, _ in top]
        example_phrase = top[0][2][0]
        joined_labels = " and ".join(labels)
        parts.append(f'hands-on {joined_labels} evidence (career history mentions "{example_phrase}")')

    _PROD_LABEL = {
        "built_production_system": "built production systems", "owned_architecture": "owned architecture",
        "deployed_to_users": "deployed to production", "scaling_optimization": "scaled/optimized systems",
        "monitoring_evaluation": "on-call/monitoring ownership", "current_hands_on_engineer": "currently hands-on IC",
    }
    prod_breakdown = subs["production_engineering"]["evidence"].get("breakdown", {})
    prod_hits = [k for k, v in prod_breakdown.items() if v.get("earned", 0) > 0]
    if prod_hits:
        parts.append(", ".join(_PROD_LABEL.get(k, k) for k in prod_hits[:3]))

    matched_skills = subs["jd_skill_alignment"]["evidence"].get("matched", {})
    if matched_skills:
        names = [s.replace("_", " ") for s in list(matched_skills.keys())[:4]]
        parts.append(f"JD-aligned skills: {', '.join(names)}")

    edu = subs["education"]
    if edu["max"] and edu["score"] / edu["max"] >= 0.75:
        edu_bd = edu["evidence"].get("breakdown", {})
        if edu_bd.get("tier_1_2_institute", {}).get("earned", 0) > 0:
            parts.append("Tier-1/2 institute background")
        else:
            parts.append("strong CS/AI/ML education background")

    company = subs["company_context"]
    if company["max"] and company["score"] / company["max"] >= 0.75:
        parts.append("AI/product company background")

    order = ["technical_substance", "jd_skill_alignment", "production_engineering",
              "behavioral_hireability", "career_progression", "company_context", "education"]
    _ALL_LABEL = {**_TECH_LABEL, "jd_skill_alignment": "skill alignment",
                   "production_engineering": "production engineering", "behavioral_hireability": "hireability signals",
                   "career_progression": "career progression", "company_context": "company context",
                   "education": "education", "technical_substance": "technical substance"}
    fracs = [(k, subs[k]["score"] / subs[k]["max"] if subs[k]["max"] else 0.0) for k in order]
    weak_k, weak_f = min(fracs, key=lambda x: x[1])
    

    _FLAG_LABEL = {
        "keyword_stuffing": "skills list not well corroborated by career history",
        "contradictory_career_history": "minor overlapping role dates",
        "impossible_timeline_soft": "tight timeline vs degree completion",
        "extremely_long_notice_period": "long notice period",
        "no_recent_activity": "inactive recently",
        "self_rated_skill_exceeds_measured": "self-rated skills exceed measured assessment",
    }
    if penalty_evidence["items"]:
        flag_text = "; ".join(_FLAG_LABEL.get(k, k) for k in penalty_evidence["items"].keys())
        parts.append(f"risk flag: {flag_text}")

   
    flag_part = parts.pop() if (penalty_evidence["items"] and parts and parts[-1].startswith("risk flag:")) else None
    parts = [p for p in parts if p][:6]
    if flag_part:
        parts.append(flag_part)
    reasoning = "; ".join(parts) if parts else "Scored on available signals."
    return reasoning[0].upper() + reasoning[1:], (weak_k, weak_f)


def score_candidate_layered(candidate, config):
    """
        Full Doc-4 additive scoring for one candidate (assumes it already passed the Cell 3 honeypot gate). Returns a result dict.
    """
    
    tech_score, tech_ev = score_technical_substance(candidate, config)
    skill_score, skill_ev = score_jd_skill_alignment(candidate, config)
    prod_score, prod_ev = score_production_engineering(candidate, config)
    behav_score, behav_ev = score_behavioral_hireability(candidate, config)
    career_score, career_ev = score_career_progression(candidate, config)
    company_score, company_ev = score_company_context(candidate, config)
    edu_score, edu_ev = score_education(candidate, config)

    penalty, penalty_ev = score_risk_penalty(candidate, config, tech_score, skill_score)

    subs = {
        "technical_substance": {"score": tech_score, "max": config["layer_points"]["technical_substance"], "evidence": tech_ev},
        "jd_skill_alignment": {"score": skill_score, "max": config["layer_points"]["jd_skill_alignment"], "evidence": skill_ev},
        "production_engineering": {"score": prod_score, "max": config["layer_points"]["production_engineering"], "evidence": prod_ev},
        "behavioral_hireability": {"score": behav_score, "max": config["layer_points"]["behavioral_hireability"], "evidence": behav_ev},
        "career_progression": {"score": career_score, "max": config["layer_points"]["career_progression"], "evidence": career_ev},
        "company_context": {"score": company_score, "max": config["layer_points"]["company_context"], "evidence": company_ev},
        "education": {"score": edu_score, "max": config["layer_points"]["education"], "evidence": edu_ev},
    }

    raw_total = sum(v["score"] for v in subs.values()) + penalty
    final_score = max(0.0, min(100.0, raw_total))

    reasoning, (weak_k, weak_f) = _build_reasoning_layered(candidate, subs, penalty_ev)

    return {
        "candidate_id": candidate.get("candidate_id"),
        "score": round(final_score, 2),
        "reasoning": reasoning,
        "breakdown": {k: round(v["score"], 2) for k, v in subs.items()},
        "risk_penalty": round(penalty, 2),
        "risk_flags": list(penalty_ev["items"].keys()),
        "concern": _build_concern(candidate, subs, weak_k, weak_f),
        "weak_frac": round(weak_f, 3),
    }


_KEY_JD_SKILL_LABELS = {
    "python": "Python", "retrieval": "retrieval", "vector_db": "vector DB tools",
    "llm_finetuning": "LLM fine-tuning", "ranking": "ranking", "evaluation_metrics": "evaluation metrics",
}


def _build_concern(candidate, subs, weak_k, weak_f):
    """
        One grounded concern sentence for the candidate's WEAKEST layer -- every claim traceable to a specific field, so Stage 4's hallucination check holds. Appended rank-aware by the pipeline runner, not here.
    """

    sig = candidate.get("redrob_signals", {}) or {}
    profile = candidate.get("profile", {}) or {}

    if weak_k == "jd_skill_alignment":
        missing = subs["jd_skill_alignment"]["evidence"].get("missing_key", [])
        if missing:
            labels = [_KEY_JD_SKILL_LABELS[k] for k in missing[:3]]
            return f"JD skills not evidenced: {', '.join(labels)}"
        return "partial JD skill coverage"
    if weak_k == "technical_substance":
        return "thin hands-on retrieval/embedding evidence in career history"
    if weak_k == "production_engineering":
        prod_bd = subs["production_engineering"]["evidence"].get("breakdown", {})
        if any(v.get("hedged") for v in prod_bd.values() if isinstance(v, dict)):
            return ("production claims use hedged language (assisted/exposure), "
                    "suggesting support work rather than full ownership")
        if any(v.get("earned", 0) > 0 for v in prod_bd.values() if isinstance(v, dict)):
            return "production evidence is narrow, not end-to-end ownership"
        return "little production ownership or deployment evidence"
    if weak_k == "behavioral_hireability":
        rr = sig.get("recruiter_response_rate")
        if isinstance(rr, (int, float)) and rr <= 0.35:
            return f"low recruiter response rate ({rr:.2f})"
        return "weak platform engagement signals"
    if weak_k == "career_progression":
        years = profile.get("years_of_experience")
        if isinstance(years, (int, float)) and not (5 <= years <= 9):
            return f"{years:.1f}y experience sits outside the JD's 5-9y target band"
        return "flat title progression across roles"
    if weak_k == "company_context":
        if subs["company_context"]["evidence"].get("is_services_company"):
            return "current role is at a services firm, not a product company"
        return "no confirmed AI/product company background"
    if weak_k == "education":
        return "no Tier-1/2 institute or CS/AI/ML degree listed"
    return ""


print("Layer 9 (Risk Penalty, 0 to -10) + master scoring function loaded.")

Layer 9 (Risk Penalty, 0 to -10) + master scoring function loaded.


In [12]:
import time as _time_pipeline


def run_layered_pipeline(candidates, config):
    t_start = _time_pipeline.time()

    survivors = []
    disqualified = 0
    for cand in candidates:
        if run_honeypot_gate(cand, config)[0]:
            survivors.append(cand)
        else:
            disqualified += 1
    processed = len(candidates)
    print(f"  honeypot filter: {len(survivors):,} survive, {disqualified:,} dropped "
          f"({100*disqualified/processed:.1f}%)")

    t_score = _time_pipeline.time()
    scored = [score_candidate_layered(cand, config) for cand in survivors]
    print(f"  scored {len(scored):,} candidates in {_time_pipeline.time()-t_score:.1f}s")

    
    scored.sort(key=lambda r: (-r["score"], r["candidate_id"]))

    top = scored[:config["top_n"]]
    for i, r in enumerate(top, 1):
        r["rank"] = i

    
    for r in top:
        concern = r.get("concern") or ""
        wf = r.get("weak_frac", 1.0)
        if r["rank"] <= 10:
            
            if concern and wf < 0.35:
                r["reasoning"] += f"; minor concern: {concern}"
        elif r["rank"] <= 50:
            if concern and wf < 0.6:
                r["reasoning"] += f"; concern: {concern}"
        else:
            if concern:
                r["reasoning"] += f"; gap: {concern}"
            if r["rank"] >= 90:
                r["reasoning"] += " -- borderline top-100 inclusion"

    elapsed = _time_pipeline.time() - t_start
    stats = {
        "total_processed": processed, "scored": len(scored), "disqualified": disqualified,
        "disqualified_pct": round(100*disqualified/processed, 2) if processed else 0,
        "top_n_returned": len(top), "elapsed_seconds": round(elapsed, 1),
        "rate_per_sec": round(processed/elapsed, 0) if elapsed else 0,
    }
    return top, stats



print(f"Scoring {len(all_candidates):,} candidates (Doc-4 additive layers)...\n")
top_results, run_stats = run_layered_pipeline(all_candidates, CONFIG)

print("\n" + "=" * 50)
for k, v in run_stats.items():
    print(f"  {k}: {v}")

if _MISSING_BEHAVIORAL_FIELDS:
    print(f"\n  WARNING: these guessed behavioral fields were never found -- "
          f"confirm field names and re-run Cell 8:")
    print(f"    {sorted(_MISSING_BEHAVIORAL_FIELDS)}")

print("\nTop 10:")
for r in top_results[:10]:
    print(f"  #{r['rank']:>3} {r['candidate_id']} score={r['score']:.2f} | {r['reasoning']}")
    print(f"       breakdown={r['breakdown']}  risk_penalty={r['risk_penalty']}  flags={r['risk_flags']}")

Scoring 100,000 candidates (Doc-4 additive layers)...



  honeypot filter: 97,142 survive, 2,858 dropped (2.9%)
  scored 97,142 candidates in 162.4s

  total_processed: 100000
  scored: 97142
  disqualified: 2858
  disqualified_pct: 2.86
  top_n_returned: 100
  elapsed_seconds: 226.7
  rate_per_sec: 441.0

Top 10:
  #  1 CAND_0008425 score=74.00 | 7.8y experience, currently Senior NLP Engineer; hands-on retrieval/search and embeddings evidence (career history mentions "retrieval"); built production systems, owned architecture, deployed to production; JD-aligned skills: python, retrieval, pgvector, lora; Tier-1/2 institute background; AI/product company background
       breakdown={'technical_substance': 24.0, 'jd_skill_alignment': 17.0, 'production_engineering': 7.5, 'behavioral_hireability': 9.0, 'career_progression': 8.0, 'company_context': 4.5, 'education': 4.0}  risk_penalty=0.0  flags=[]
  #  2 CAND_0046064 score=73.76 | 8.9y experience, currently Senior NLP Engineer; hands-on embeddings and vector databases evidence (career history me

In [ ]:
import csv as _csv

OUTPUT_PATH = "team_ai_quartet.csv"  # rename to your required filename

_RUBRIC_MAX = sum(CONFIG["layer_points"].values())  # = 97


def _validate_submission(rows, config):
    """
        Checks rows against the spec: exactly top_n rows, ranks 1..N contiguous, score non-increasing, reasoning non-empty. Raises on any problem.
    """
    
    top_n = config["top_n"]
    problems = []

    if len(rows) != top_n:
        problems.append(f"expected {top_n} rows, got {len(rows)}")

    ranks = [r["rank"] for r in rows]
    if ranks != list(range(1, len(rows) + 1)):
        problems.append("ranks are not a clean 1..N sequence")

    for i in range(1, len(rows)):
        if rows[i]["score"] > rows[i - 1]["score"]:
            problems.append(
                f"score increases at rank {rows[i]['rank']} "
                f"({rows[i-1]['score']} -> {rows[i]['score']})"
            )
            break

    empty_reasons = [r["rank"] for r in rows if not (r.get("reasoning") or "").strip()]
    if empty_reasons:
        problems.append(f"empty reasoning at ranks: {empty_reasons[:5]}")

    if problems:
        raise ValueError("Submission validation failed:\n  - " + "\n  - ".join(problems))
    return True


def write_submission_csv(top_results, config, output_path=OUTPUT_PATH):
    """
        Validates then writes ranked results to CSV in required column order. Scores are written as raw_score / 97, rounded to 4 decimals.
    """

    columns = config["output_columns"]
    _validate_submission(top_results, config)

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = _csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for r in top_results:
            writer.writerow({
                "rank": r["rank"],
                "candidate_id": r["candidate_id"],
                "score": round(r["score"] / _RUBRIC_MAX, 4),
                "reasoning": r["reasoning"],
            })

    print(f"Submission written to {output_path}")
    print(f"  {len(top_results)} rows, columns: {columns}")
    print(f"  scores normalized: raw/{_RUBRIC_MAX} (rank 1 = "
          f"{top_results[0]['score']}/{_RUBRIC_MAX} = {top_results[0]['score']/_RUBRIC_MAX:.4f})")
    return output_path


# ---- Write it ----
_path = write_submission_csv(top_results, CONFIG)

print("\nFirst 5 rows of the CSV:")
with open(_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i > 5:
            break
        print("  " + line.rstrip())

Submission written to submission.csv
  100 rows, columns: ['candidate_id', 'rank', 'score', 'reasoning']
  scores normalized: raw/97 (rank 1 = 74.0/97 = 0.7629)

First 5 rows of the CSV:
  candidate_id,rank,score,reasoning
  CAND_0008425,1,0.7629,"7.8y experience, currently Senior NLP Engineer; hands-on retrieval/search and embeddings evidence (career history mentions ""retrieval""); built production systems, owned architecture, deployed to production; JD-aligned skills: python, retrieval, pgvector, lora; Tier-1/2 institute background; AI/product company background"
  CAND_0046064,2,0.7604,"8.9y experience, currently Senior NLP Engineer; hands-on embeddings and vector databases evidence (career history mentions ""embedding""); built production systems, owned architecture, deployed to production; JD-aligned skills: python, retrieval, pinecone, lora; Tier-1/2 institute background; AI/product company background"
  CAND_0081846,3,0.7574,"6.7y experience, currently Lead AI Engineer; hands-o